# data_clean: cleaning the referral records

This notebook loads the referral records from the data custodian's secure database and prepares them
for splitting. The steps are:

1. Drop the columns outside the analysis
2. Drop the columns with more than 40 percent missing values
3. Drop duplicate patients, keeping the most recent referral, then drop RequestDate
4. One-hot encode the categorical variables
5. Scale the ordinal variables to 0 to 1
6. Recode Gender to 0 and 1
7. Prepare the MHQ items: fill missing FQ values with 0 and reverse-code SIAS items 5, 9 and 11
8. Merge treatment class 5 into class 4, then remove patients with no recorded treatment class
9. Drop rows with missing values
10. Save `sx_clean_df.csv`, holding the individual MHQ items
11. Sum the MHQ items into questionnaire scores, each divided by a fixed maximum score
12. Save `sx_clean_Total_df.csv`, holding the summed scores

The source records come from the data custodian's secure database, so this notebook runs only with
that access, and a local `config_secure.py` file holds the credentials. The two output files go into
`data/raw_data/`.

# Imports

In [ ]:
import pymssql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
from config_secure import password   # local credentials file, held by the data custodian and not distributed

In [ ]:
conn = pymssql.connect(
    server="<SERVER>",
    port=1433,
    user="<USERNAME>",
    password = password,
    database="<DATABASE>")
cursor = conn.cursor()
cursor.execute("SELECT @@VERSION")

cursor.execute("SELECT * FROM SurveyXact")
rows = cursor.fetchall()
cols = [col[0] for col in cursor.description]
sx_df = pd.DataFrame(rows, columns=cols)

sx_df.info()

# 1. Drop columns not required for analysis 

In [ ]:
# Drop the columns the analysis leaves out
sx_2_df = sx_df.drop(['LetterTemplate',
                      'ApplyFor',
                      'UseOfDrugs', # majority is 'no'
                      'Ocd', 'Bipolar', 'Schizophrenia', 'Ptsd', 'PersonalityDisorder',  # the classification uses the MHQ items
                      'TerminationType', 
                      'ScreeningTypeResult',
                      'TreatmentStartDate',
                      'Motivation',
                      'ExpectationTreatment', 
                      'ExpectationInternetBasedTherapy', 
                      'WhyInternetBasedTherapy',
                      'ExplainSymptoms',
                      'Comment', 
                      'HasPhobia',
                      'MansaTwentyFour', 
                      'MansaTen', 
                      'MansaTwentyFive',], axis=1)
sx_2_df.info()

# 2. Drop columns with missing-ness >40%

In [ ]:
# Step 1: total number of rows
total_rows = len(sx_2_df)

# Step 2: count non-missing entries per column
# .notna() returns True for non-NaN/non-None values; sum() counts them.
non_null_counts = sx_2_df.notna().sum()

# Step 3: find the maximum number of non-missing values in any column
# This identifies the "most complete" column(s).
max_complete = non_null_counts.max()

# Step 4: identify which column(s) have that maximum completeness
# This is useful to know what best-case completeness looks like.
most_complete_cols = non_null_counts[non_null_counts == max_complete].index.tolist()

print(f"Total rows in dataframe (X): {total_rows}")
print(f"Maximum non-missing entries in any column: {max_complete}")
print("Column(s) with largest complete data:")
for col in most_complete_cols:
    print(f"  - {col}")

# Step 5: compute missing counts per column
# Missing rows are the gap between total rows and non-missing entries.
missing_counts = total_rows - non_null_counts

# Step 6: compute percentage missingness relative to total X
# This gives the percentage of all rows missing in each column.
percent_missing = (missing_counts / total_rows) * 100

# Step 7: build a summary table
# This table will help decide systematically which columns to drop.
sx_nanPerct_df = pd.DataFrame({
    "non_missing": non_null_counts,                   # how many valid values
    "missing": missing_counts,                       # how many NaN/None
    "% missingness of total X": percent_missing  # percentage of all rows
})

# Step 8: sort by % missingness (most missing first)
sx_nanPerct_df = sx_nanPerct_df.sort_values("% missingness of total X", ascending=False)

print("\nColumn-wise missingness summary:")
print(sx_nanPerct_df.round(2).head(60))  # round percentages for neat printing

In [ ]:
print(sx_nanPerct_df.round(2).tail(60))  # round percentages for neat printing


In [ ]:
# Drop the columns above the 40 percent missingness threshold, and PhqTen
sx_cols_keep_df = sx_2_df.drop(['DiagnosedEarlier', 'PhqTen',
                                'NumberTreatment', 'NumberChildren',
                                'WhoOne', 'WhoTwo', 'WhoThree', 'WhoFour','WhoFive',
                                'EqFiveDOne','EqFiveDTwo', 'EqFiveDThree', 'EqFiveDFour', 
                                'EqFiveDFive'], axis=1)
                                
sx_cols_keep_df.info()

# 3. Drop duplicate patients and drop RequestDate

In [ ]:
sx_cols_keep_df['RequestDate'] = pd.to_datetime(sx_cols_keep_df['RequestDate'])
sx_sort_df = sx_cols_keep_df.sort_values(by='RequestDate', ascending=False)

# Drop duplicate ParticipantID rows, keeping the most recent referral
sx_unique_df = sx_sort_df.drop_duplicates(subset='ParticipantID', keep='first')

sx_unq_df = sx_unique_df.drop(['RequestDate'], axis=1)
# Display the resulting DataFrame
print(f"Dataframe 'sx_sort_df' size before duplicate ParticipantID removed : {sx_sort_df.shape}") 
print(f"Dataframe 'sx_unq_df' size after duplicate ParticipantID removed : {sx_unq_df.shape}") 


In [ ]:
# Choose the column 
col5 = "Income"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_unq_df[col5].value_counts(dropna=False)

print(f"\nValue counts for column: {col5}")
print("-" * (len(col5) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_unq_df)}")

# Check for missing values in that column
missing_count = len(sx_unq_df) - total_count
print(f"Missing values in '{col5}': {missing_count}")

In [ ]:
# Choose the column 
col5 = "Income"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_unq_df[col5].value_counts(dropna=False)

print(f"\nValue counts for column: {col5}")
print("-" * (len(col5) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_unq_df)}")

# Check for missing values in that column
missing_count = len(sx_unq_df) - total_count
print(f"Missing values in '{col5}': {missing_count}")

In [ ]:
# Choose the column 
col6 = "DiagnosedWithAnxietyOrDepression"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_unq_df[col6].value_counts(dropna=False)

print(f"\nValue counts for column: {col6}")
print("-" * (len(col6) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_unq_df)}")

# Check for missing values in that column
missing_count = len(sx_unq_df) - total_count
print(f"Missing values in '{col6}': {missing_count}")

In [ ]:
# Choose the column 
col7 = "AlcoholIntake"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_unq_df[col7].value_counts(dropna=False)

print(f"\nValue counts for column: {col7}")
print("-" * (len(col7) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_unq_df)}")

# Check for missing values in that column
missing_count = len(sx_unq_df) - total_count
print(f"Missing values in '{col7}': {missing_count}")

# 4. One hot encode categorical variables 

In [ ]:
# List of categorical variables to one-hot encode with their prefixes
categorical_variables = {
    'CivilStatus': 'oh_civil',
    'Income': 'oh_income',
    'DiagnosedWithAnxietyOrDepression': 'oh_dad',
}

# One-hot encode the specified categorical variables with custom prefixes
def one_hot_encode_with_prefix(df, variable_prefixes):
    """
    One-hot encodes the specified categorical variables in the DataFrame with custom prefixes.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data.
        variable_prefixes (dict): A dictionary where keys are variable names and values are prefixes.

    Returns:
        pd.DataFrame: A new DataFrame with the one-hot encoded variables.
    """
    for variable, prefix in variable_prefixes.items():
        df = pd.get_dummies(df, columns=[variable], prefix=prefix, drop_first=False)
    return df
# Apply the one hot encoding to the df
sx_oh_df = one_hot_encode_with_prefix(sx_unq_df, categorical_variables)

# 5. Scale ordinal variables

In [ ]:
#List of ordinal variables to scale
ordinal_variables = ['Education', 'AlcoholIntake', 'CurrentMedication']

# Function to scale ordinal variables to 0-1 range
def scale_ordinal_variables(df, variables):
    """
    Scales ordinal variables to the range 0-1 based on the number of unique values.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the data.
        variables (list): List of variable names to scale.

    Returns:
        pd.DataFrame: A new DataFrame with scaled ordinal variables.
    """
    df_scaled = df.copy()  # Create a copy to avoid modifying the original DataFrame
    for variable in variables:
        if variable in df_scaled.columns:
            # Find the number of unique values
            max_value = df_scaled[variable].nunique()
            print(f"Variable '{variable}' has {max_value} unique values.")  # Print max_value
            # Scale the values by dividing by the number of unique values
            df_scaled[variable] = df_scaled[variable] / max_value
        else:
            print(f"Variable '{variable}' not found in the DataFrame.")
    return df_scaled
    
# Apply the scaling to the df
sx_scaled_df = scale_ordinal_variables(sx_oh_df, ordinal_variables)


In [ ]:
# Choose the column 
col7 = "AlcoholIntake"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_scaled_df[col7].value_counts(dropna=False)

print(f"\nValue counts for column: {col7}")
print("-" * (len(col7) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_scaled_df)}")

# Check for missing values in that column
missing_count = len(sx_scaled_df) - total_count
print(f"Missing values in '{col7}': {missing_count}")

In [ ]:
# Choose the column 
col8 = "Education"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_scaled_df[col8].value_counts(dropna=False)

print(f"\nValue counts for column: {col8}")
print("-" * (len(col8) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_scaled_df)}")

# Check for missing values in that column
missing_count = len(sx_scaled_df) - total_count
print(f"Missing values in '{col8}': {missing_count}")

# 6. Normalize Gender  

In [ ]:
# Print gender counts before normalizing
sx_scaled_df['Gender'].value_counts()

sx_gen_norm_df = sx_scaled_df.copy()

# Normalise the Gender variable
sx_gen_norm_df.loc[:, 'Gender'] = sx_gen_norm_df['Gender'].replace({1: 0, 2: 1})

# Print gender counts after normalizing
sx_gen_norm_df['Gender'].value_counts()


In [ ]:
# Choose the column 
col3 = "Gender"  # column to inspect

# Step 1: Compute the value counts
# dropna=False ensures NaN values are included in the count
vc = sx_gen_norm_df[col3].value_counts(dropna=False)

print(f"\nValue counts for column: {col3}")
print("-" * (len(col3) + 20))

# Step 2: Print counts neatly
for value, count in vc.items():
    print(f"  Value: {value} --> Count: {count}")

# Step 3: Sum all the counts
total_count = vc.sum()

print(f"\nSum of all value counts: {total_count}")
print(f"Total rows in dataframe: {len(sx_gen_norm_df)}")

# Check for missing values in that column
missing_count = len(sx_gen_norm_df) - total_count
print(f"Missing values in '{col3}': {missing_count}")

In [ ]:
sx_gen_norm_df.info()

# 7. Normalize MHQ’s 

In [ ]:
sx_mhqNorm_df = sx_gen_norm_df.copy()

# handle missing 'Fq' values - fill NaN with 0 - 
sx_mhqNorm_df['FqOne'] = sx_mhqNorm_df['FqOne'].fillna(0)
sx_mhqNorm_df['FqSeventeenOther'] = sx_mhqNorm_df['FqSeventeenOther'].fillna(0)
sx_mhqNorm_df['FqTwentyFourOther'] = sx_mhqNorm_df['FqTwentyFourOther'].fillna(0)

# reverse code Sias q's 
sx_mhqNorm_df.loc[:, 'Sias5R'] = 4 - sx_mhqNorm_df['siasFive']    # Reverse Coded
sx_mhqNorm_df.drop(['siasFive'], inplace=True, axis=1)

sx_mhqNorm_df.loc[:, 'Sias9R'] = 4 - sx_mhqNorm_df['siasNine']    # Reverse Coded
sx_mhqNorm_df.drop(['siasNine'], inplace=True, axis=1)

sx_mhqNorm_df.loc[:, 'Sias11R'] = 4 - sx_mhqNorm_df['SiasEleven']    # Reverse Coded
sx_mhqNorm_df.drop(['SiasEleven'], inplace=True, axis=1)


# 8. Handle missing TreatmentType values and drop ‘0’

In [ ]:
print("Before normalizing -----")
print(sx_mhqNorm_df['TreatmentType'].value_counts())

sx_trTy_Nan_df = sx_mhqNorm_df.copy()

sx_trTy_Nan_df['TreatmentType'] = sx_trTy_Nan_df['TreatmentType'].fillna(0)

print("\nAfter normalizing ------")
print(sx_trTy_Nan_df['TreatmentType'].value_counts())

sx_trTy_comb_df = sx_trTy_Nan_df.copy()
sx_trTy_comb_df['TreatmentType'] = sx_trTy_comb_df['TreatmentType'].replace(5.0, 4.0)
print("\nAfter combining 4+5 ------")
print(sx_trTy_comb_df['TreatmentType'].value_counts())

print(f"0 - Group course / ")
print(f"1 - DEPRESSION / Hospitalization in a psychiatric ward")
print(f"2 - PANIC DISORDER / Psychologist (individual interview")
print(f"3 - SOCIAL PHOBIA / Psychiatrist")
print(f"4 - SPECIFIC PHOBIA / Internet treatment")


In [ ]:
# balance: drops a share frac of the rows where col equals value. The call below uses
# frac = 1.0, so it drops every row with TreatmentType 0.
def balance(df1, col, value, frac):
    '''Reduce df['col']==value by frac'''
    
    # drop overrepresented values
    z_df = df1[df1[col] == value]
    drop_ind = z_df.sample(frac = frac, random_state=42)   
    df1 = df1.drop(drop_ind.index)
    return df1

In [ ]:
sx_tTclass_df = balance(sx_trTy_comb_df, 'TreatmentType', 0, 1.0)


In [ ]:
print("After dropping -----")
print(sx_tTclass_df['TreatmentType'].value_counts())

# 9. Drop NaN row

In [ ]:
nan_counts = sx_tTclass_df.isna().sum()
print("Columns and their NaN counts:")
nan_counts.head(44)

In [ ]:
nan_counts.tail(44)

In [ ]:
sx_clean_df = sx_tTclass_df.dropna()

print("Columns and their NaN counts AFTER dropping:")
nan_counts2 = sx_clean_df.isna().sum()

nan_counts2.head(44)

In [ ]:
nan_counts2.tail(44)

# 10. Save data - individual scores

In [ ]:
sx_clean_df.to_csv('sx_clean_df.csv', index=True)
print(f"Data with individual scores saved as 'sx_clean_df.csv'")

# 11. Sum MHQ and replace individual score columns 

In [ ]:
sx_clean_T_df = sx_clean_df.copy()

In [ ]:
# PHQ
# Calculate 'phq' for each row
sx_clean_T_df['phq'] = (sx_clean_T_df['PhqOne'] 
                      + sx_clean_T_df['PhqTwo']
                      + sx_clean_T_df['PhqThree'] 
                      + sx_clean_T_df['PhqFour'] 
                      + sx_clean_T_df['PhqFive'] 
                      + sx_clean_T_df['PhqSix'] 
                      + sx_clean_T_df['PhqSeven'] 
                      + sx_clean_T_df['PhqEight'] 
                      + sx_clean_T_df['PhqNine']) / 27
sx_clean_T_df['phq'].value_counts()

In [ ]:
# GAD
# Calculate 'gad' for each row
sx_clean_T_df['gad'] = (sx_clean_T_df['GadOne'] 
                      + sx_clean_T_df['GadTwo'] 
                      + sx_clean_T_df['GadThree'] 
                      + sx_clean_T_df['GadFour'] 
                      + sx_clean_T_df['GadFive'] 
                      + sx_clean_T_df['GadSix'] 
                      + sx_clean_T_df['gadSeven']) / 21
sx_clean_T_df['gad'].value_counts()

In [ ]:
# SIAS
# Calculate 'sias' for each row
sx_clean_T_df['sias'] = (sx_clean_T_df['SiasOne'] 
                       + sx_clean_T_df['SiasTwo']
                       + sx_clean_T_df['SiasThree'] 
                       + sx_clean_T_df['SiasFour'] 
                       + sx_clean_T_df['Sias5R'] 
                       + sx_clean_T_df['SiasSix'] 
                       + sx_clean_T_df['SiasSeven'] 
                       + sx_clean_T_df['SiasEight'] 
                       + sx_clean_T_df['Sias9R'] 
                       + sx_clean_T_df['SiasTen'] 
                       + sx_clean_T_df['Sias11R'] 
                       + sx_clean_T_df['SiasTwelv'] 
                       + sx_clean_T_df['SiasThirteen'] 
                       + sx_clean_T_df['SiasFourteen'] 
                       + sx_clean_T_df['SiasFifteen'] 
                       + sx_clean_T_df['SiasSixteen'] 
                       + sx_clean_T_df['SiasSeventeen'] 
                       + sx_clean_T_df['SiasEighteen'] 
                       + sx_clean_T_df['SiasNineteen']
                       + sx_clean_T_df['SiasTwenty']) / 80
sx_clean_T_df['sias'].value_counts()

In [ ]:
# PDSS

# Calculate 'pdss' for each row
sx_clean_T_df['pdss'] = (sx_clean_T_df['PdssOne'] 
                       + sx_clean_T_df['PdssTwo'] 
                      + sx_clean_T_df['PdssThree'] 
                       + sx_clean_T_df['PdssFour'] 
                      + sx_clean_T_df['PdssFive'] 
                       + sx_clean_T_df['PdssSix'] 
                      + sx_clean_T_df['PdssSeven']) / 28
sx_clean_T_df['pdss'].value_counts()

In [ ]:

#FQ
# Calculate 'fq' for each row

sx_clean_T_df['fq_ago'] = (sx_clean_T_df['FqFive']
                         + sx_clean_T_df['FqSix']
                         + sx_clean_T_df['FqEight']
                         + sx_clean_T_df['FqTwelve']
                         + sx_clean_T_df['FqFifteen']) / 40
sx_clean_T_df['fq_bld'] = (sx_clean_T_df['FqTwo']
                         + sx_clean_T_df['FqFour']
                         + sx_clean_T_df['FqTen']
                         + sx_clean_T_df['FqThirteen']
                         + sx_clean_T_df['FqSixteen']) / 40
sx_clean_T_df['fq_soc'] = (sx_clean_T_df['FqThree']
                         + sx_clean_T_df['FqSeven']
                         + sx_clean_T_df['FqNine']
                         + sx_clean_T_df['FqEleven']
                         + sx_clean_T_df['FqFourteen']) / 40
sx_clean_T_df['fq_total'] = (sx_clean_T_df['fq_ago'] 
                           + sx_clean_T_df['fq_bld'] 
                           + sx_clean_T_df['fq_soc']) / 3
sx_clean_T_df['fq_main'] = sx_clean_T_df['FqOne'] / 8
sx_clean_T_df['fq_global'] = sx_clean_T_df['FqEighteen'] / 8
sx_clean_T_df['fq_anx_dep'] = (sx_clean_T_df['FqNineteen']
                             + sx_clean_T_df['FqTwenty']
                             + sx_clean_T_df['FqTwentyOne']
                             + sx_clean_T_df['FqTwentyTwo']
                             + sx_clean_T_df['FqTwentyThree']
                             + sx_clean_T_df['FqTwentyFourOther']) / 40
sx_clean_T_df['fq_ago'].value_counts()
sx_clean_T_df['fq_bld'].value_counts()
sx_clean_T_df['fq_soc'].value_counts()
sx_clean_T_df['fq_total'].value_counts()
sx_clean_T_df['fq_main'].value_counts()
sx_clean_T_df['fq_global'].value_counts()
sx_clean_T_df['fq_anx_dep'].value_counts()

In [ ]:
# Drop the columns that contributed to the 'phq' and 'gad' sums
columns_to_drop = ['PhqOne', 'PhqTwo', 'PhqThree', 'PhqFour', 'PhqFive', 'PhqSix', 'PhqSeven', 'PhqEight', 'PhqNine',
                   'GadOne', 'GadTwo', 'GadThree', 'GadFour', 'GadFive', 'GadSix', 'gadSeven',
                   'FqOne', 'FqTwo', 'FqThree', 'FqFour', 'FqFive', 'FqSix', 'FqSeven', 'FqEight', 'FqNine', 'FqTen', 
                   'FqEleven', 'FqTwelve', 'FqThirteen', 'FqFourteen', 'FqFifteen', 'FqSixteen', 'FqSeventeenOther', 
                   'FqEighteen', 'FqNineteen', 'FqTwenty', 'FqTwentyOne', 'FqTwentyTwo', 'FqTwentyThree', 'FqTwentyFourOther', 
                   'SiasOne', 'SiasTwo', 'SiasThree', 'SiasFour', 'Sias5R', 'SiasSix', 'SiasSeven', 'SiasEight', 'Sias9R', 
                   'SiasTen','Sias11R', 'SiasTwelv', 'SiasThirteen', 'SiasFourteen', 'SiasFifteen', 'SiasSixteen', 
                   'SiasSeventeen', 'SiasEighteen', 'SiasNineteen', 'SiasTwenty', 
                   'PdssOne','PdssTwo', 'PdssThree', 'PdssFour', 'PdssFive', 'PdssSix', 'PdssSeven']
sx_clean_Total_df = sx_clean_T_df.drop(columns=columns_to_drop)

# Display the updated DataFrame
print("sx_clean_Total_df after dropping the contributing columns {sx_clean_Total_df}")
sx_clean_Total_df.info()

# 12. Save data 

In [ ]:
sx_clean_Total_df.to_csv('sx_clean_Total_df.csv', index=True)
print(f"Data with individual scores saved as 'sx_clean_Total_df.csv'")

sx_clean_Total_df.info()

print(sx_clean_Total_df['TreatmentType'].value_counts())
sx_clean_Total_df

# 13. Final shape prints

In [ ]:
print(f"Step 0: Size of original df:-------------------------------------------------{sx_df.shape}") 
print(f"Step 1: Size after 'not needed' columns removed:-----------------------------{sx_2_df.shape}") 
print(f"Step 2: Size after missingness >40% columns; PhqTen removed:------------------{sx_cols_keep_df.shape}") 
print(f"Step 3: Size after duplicate patients removed:-------------------------------{sx_unq_df.shape}")
print(f"Step 4: Size after categorical variables o-h-e:------------------------------{sx_oh_df.shape}")
print(f"Step 5: Size after ordinal variables scaled:---------------------------------{sx_scaled_df.shape}")
print(f"Step 6: Size after gender variable normalized:-------------------------------{sx_gen_norm_df.shape}")
print(f"Step 7: Size after MHQ variables normalized:---------------------------------{sx_mhqNorm_df.shape}")
print(f"Step 8: Size after TreatmentType variable 'NaN' value rows converted to '0': {sx_trTy_comb_df.shape}")
print(f"Step 8: Size after TreatmentType variable '4'+'5' rows combined:-------------{sx_trTy_comb_df.shape}")
print(f"Step 8: Size after TreatmentType variable '0' rows removed:------------------{sx_tTclass_df.shape}")
print(f"Step 9: Size after all NaN value rows removed:-------------------------------{sx_clean_df.shape}")
print(f"Step 10: csv files saved as ----------------------------'sx_clean_df.csv'")
print(f"Step 11: Size after MHQs aggregated:-----------------------------------------{sx_clean_Total_df.shape}")
print(f"Step 12: csv files saved as ----------------------------'sx_clean_Total_df.csv'")